In [16]:
import numpy as np
import pandas as pd

In [17]:
def one_hot_encoding(labels):
    vocab_list = list(set(labels))
    length = len(labels)
    output = []
    for i in range(length):
        zeros = [0] * len(vocab_list)
        pos = vocab_list.index(labels[i])
        zeros[pos] = 1
        output.append(zeros)
    
    return output, vocab_list

def ini_parameters(inputs, num_neurons1, num_neurons2, num_neurons3, output):
    input_col = inputs.shape[1]
    output_col = output.shape[1]
    weights1 = np.random.randn(input_col, num_neurons1) * np.sqrt(2.0 / input_col)
    bias1 = np.zeros((1, num_neurons1))

    weights2 = np.random.randn(num_neurons1, num_neurons2) * np.sqrt(2.0 / num_neurons1)
    bias2 = np.zeros((1, num_neurons2))

    weights3 = np.random.randn(num_neurons2, num_neurons3) * np.sqrt(2.0 / num_neurons2)
    bias3 = np.zeros((1, num_neurons3))

    # Xavier Initialization for Softmax layer:
    output_weights = np.random.randn(num_neurons3, output_col) * np.sqrt(1.0 / num_neurons3)
    output_bias = np.zeros((1, output_col))

    return weights1, bias1, weights2, bias2, weights3, bias3, output_weights, output_bias


def ReLU(z):
    return np.maximum(0,z)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def Foward_Pass(inputs, weights1, bias1, weights2, bias2, weights3, bias3, output_weights, output_bias):
    Z1 = inputs @ weights1 + bias1
    A1 = ReLU(Z1)
    
    Z2 = A1 @ weights2 + bias2
    A2 = ReLU(Z2)

    Z3 = A2 @ weights3 + bias3
    A3 = ReLU(Z3)

    Z4 = A3 @ output_weights + output_bias
    prediction = softmax(Z4)

    return A1, A2, A3, prediction

def Loss(prediction, labels):
    clipped_pred = np.clip(prediction, 1e-15, 1 - 1e-15)
    log_pred = np.log(clipped_pred)
    loss = labels * log_pred
    avg_loss = -(np.sum(loss)/labels.shape[0])
    return avg_loss

def Back_Prop(inputs, labels, A1, A2, A3, prediction, weights2, weights3, output_weights):
    m = labels.shape[0]

    dZ4 = prediction - labels
    dB4 = np.sum(dZ4, axis=0, keepdims=True) / m
    dW4 = (A3.T @ dZ4) / m

    dZ3 = (dZ4 @ output_weights.T) * (A3 > 0)
    dB3 = np.sum(dZ3, axis=0, keepdims=True)/ m
    dW3 = (A2.T @ dZ3) / m

    dZ2 = (dZ3 @ weights3.T) * (A2 > 0)
    dB2 = np.sum(dZ2, axis=0, keepdims=True)/ m
    dW2 = (A1.T @ dZ2) / m

    dZ1 = (dZ2 @ weights2.T) * (A1 > 0)
    dB1 = np.sum(dZ1, axis=0, keepdims=True)/ m
    dW1 = (inputs.T @ dZ1) / m

    return dW1, dB1, dW2, dB2, dW3, dB3, dW4, dB4

def Train(inputs, labels, num_neurons1, num_neurons2, num_neurons3, epochs, lr):
    weights1, bias1, weights2, bias2, weights3, bias3, output_weights, output_bias = ini_parameters(inputs, num_neurons1, num_neurons2, num_neurons3, labels)

    for epoch in range(epochs):
        A1, A2, A3, prediction = Foward_Pass(inputs, weights1, bias1, weights2, bias2, weights3, bias3, output_weights, output_bias)
        
        dW1, dB1, dW2, dB2, dW3, dB3, dW4, dB4 = Back_Prop(inputs, labels, A1, A2, A3, prediction, weights2, weights3, output_weights)

        if epoch % 100 == 0:
            pred_classes = np.argmax(prediction, axis=1)
            true_classes = np.argmax(labels, axis=1)
            
            accuracy = np.mean(pred_classes == true_classes)
            
            accuracy_percentage = accuracy * 100
            
            print(f"Epoch {epoch} | Accuracy: {accuracy_percentage:.2f}%")

        weights1 -= lr * dW1
        bias1 -= lr * dB1
        
        weights2 -= lr * dW2
        bias2 -= lr * dB2

        weights3 -= lr * dW3
        bias3 -= lr * dB3

        output_weights -= lr * dW4
        output_bias -= lr * dB4

    final_pred_classes = np.argmax(prediction, axis=1)
    final_true_classes = np.argmax(labels, axis=1)
    final_accuracy = np.mean(final_pred_classes == final_true_classes) * 100

    return weights1, bias1, weights2, bias2, weights3, bias3, output_weights, output_bias, final_accuracy

In [18]:
X_fake = np.random.randn(100, 20)
fake_raw_labels = np.random.choice(['apple', 'banana', 'cherry'], size=100)
encoded_list, vocabulary = one_hot_encoding(fake_raw_labels)
Y_fake = np.array(encoded_list)


print("Starting Training")
w1, b1, w2, b2, w3, b3, ow, ob, loss_result = Train(
    inputs=X_fake,
    labels=Y_fake,
    num_neurons1=16,  
    num_neurons2=8,   
    num_neurons3=4,   
    epochs=1500,
    lr=0.05
)

print(f"\nTraining Complete! Final baseline error score: {loss_result:.2f}%")


Starting Training
Epoch 0 | Accuracy: 32.00%
Epoch 100 | Accuracy: 52.00%
Epoch 200 | Accuracy: 55.00%
Epoch 300 | Accuracy: 58.00%
Epoch 400 | Accuracy: 68.00%
Epoch 500 | Accuracy: 72.00%
Epoch 600 | Accuracy: 74.00%
Epoch 700 | Accuracy: 77.00%
Epoch 800 | Accuracy: 85.00%
Epoch 900 | Accuracy: 88.00%
Epoch 1000 | Accuracy: 89.00%
Epoch 1100 | Accuracy: 89.00%
Epoch 1200 | Accuracy: 92.00%
Epoch 1300 | Accuracy: 92.00%
Epoch 1400 | Accuracy: 92.00%

Training Complete! Final baseline error score: 92.00%


In [19]:
df = pd.read_csv("Iris.csv")

raw_features = df.iloc[:, 1:-1].values
raw_labels = df.iloc[:, -1].values

encoded_labels_list, vocab = one_hot_encoding(raw_labels)
labels = np.array(encoded_labels_list)
inputs = np.array(raw_features, dtype=float)

w1, b1, w2, b2, w3, b3, ow, ob, loss_result = Train(
    inputs = inputs,
    labels = labels,
    num_neurons1=32,
    num_neurons2=16,
    num_neurons3=8,
    epochs=1000,
    lr = 0.01

 )



Epoch 0 | Accuracy: 31.33%
Epoch 100 | Accuracy: 88.67%
Epoch 200 | Accuracy: 97.33%
Epoch 300 | Accuracy: 98.00%
Epoch 400 | Accuracy: 98.00%
Epoch 500 | Accuracy: 93.33%
Epoch 600 | Accuracy: 98.00%
Epoch 700 | Accuracy: 98.67%
Epoch 800 | Accuracy: 98.67%
Epoch 900 | Accuracy: 98.67%


In [20]:
def Predict(new_inputs, true_labels, trained_parameters, vocab_list):
    w1, b1, w2, b2, w3, b3, ow, ob = trained_parameters
    
    _, _, _, prediction_probabilities = Foward_Pass(new_inputs, w1, b1, w2, b2, w3, b3, ow, ob)
    predicted_indices = np.argmax(prediction_probabilities, axis=1)
    predicted_labels = [vocab_list[idx] for idx in predicted_indices]
    
    print(f"{'Sample':<8} | {'Network Guess':<18} | {'True Label':<18} | {'Match?':<6}")
    print("-" * 58)
    
    for i in range(len(predicted_labels)):
        guess = predicted_labels[i]
        actual = true_labels[i]
        match = "YES" if guess == actual else "NO"
        print(f"{i:<8} | {guess:<18} | {actual:<18} | {match:<6}")
        
    return predicted_labels



In [ ]:
import pandas as pd
import numpy as np

# 1. Load and Shuffle
df = pd.read_csv("Iris.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Extract and Prepare
raw_features = df.iloc[:, 1:-1].values
raw_labels = df.iloc[:, -1].values
encoded_labels_list, vocab = one_hot_encoding(raw_labels)

X = np.array(raw_features, dtype=float)
Y = np.array(encoded_labels_list)

# 3. Calculate Split Point (75% training)
split_idx = int(0.75 * len(X))

X_train, X_test = X[:split_idx], X[split_idx:]
Y_train, Y_test = Y[:split_idx], Y[split_idx:]
raw_labels_test = raw_labels[split_idx:]

# 4. Train on the 75% chunk
w1, b1, w2, b2, w3, b3, ow, ob, final_acc = Train(
    inputs=X_train,
    labels=Y_train,
    num_neurons1=32,
    num_neurons2=16,
    num_neurons3=8,
    epochs=1000,
    lr=0.01
)

# 5. Predict on the unseen 25% chunk
my_model = (w1, b1, w2, b2, w3, b3, ow, ob)
predictions = Predict(X_test, raw_labels_test, my_model, vocab)

test_accuracy = np.mean(np.array(predictions) == raw_labels_test) * 100
print(f"\nFinal Test Set Accuracy: {test_accuracy:.2f}%")

Epoch 0 | Accuracy: 31.25%
Epoch 100 | Accuracy: 41.96%
Epoch 200 | Accuracy: 96.43%
Epoch 300 | Accuracy: 90.18%
Epoch 400 | Accuracy: 93.75%
Epoch 500 | Accuracy: 93.75%
Epoch 600 | Accuracy: 94.64%
Epoch 700 | Accuracy: 95.54%
Epoch 800 | Accuracy: 95.54%
Epoch 900 | Accuracy: 95.54%
Sample   | Network Guess      | True Label         | Match?
----------------------------------------------------------
0        | Iris-versicolor    | Iris-versicolor    | YES   
1        | Iris-virginica     | Iris-virginica     | YES   
2        | Iris-setosa        | Iris-setosa        | YES   
3        | Iris-virginica     | Iris-virginica     | YES   
4        | Iris-versicolor    | Iris-versicolor    | YES   
5        | Iris-virginica     | Iris-virginica     | YES   
6        | Iris-versicolor    | Iris-versicolor    | YES   
7        | Iris-versicolor    | Iris-versicolor    | YES   
8        | Iris-versicolor    | Iris-versicolor    | YES   
9        | Iris-setosa        | Iris-setosa        | 